<a href="https://colab.research.google.com/github/sairiamu/ML-DL-collection/blob/main/whisper_voice_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Voice System: Speech ↔ Text (Colab)

```
 mic / audio file ──► [ faster-whisper (STT) ] ──► text
 text ──► [ Kokoro-82M (TTS) ] ──► audio (24 kHz wav)
```

**Runtime → Change runtime type → T4 GPU** (works on CPU too, just slower).

Sections: 1. Install · 2. STT engine · 3. TTS engine · 4. Mic recording in Colab · 5. Round-trip test · 6. Gradio UI · 7. Optional: FastAPI service + Swahili TTS

## 1. Install

In [ ]:
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
!pip -q install faster-whisper kokoro soundfile gradio numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 73.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


## 2. Speech → Text engine (faster-whisper)

In [ ]:
import torch, numpy as np, soundfile as sf
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
# GPU: large-v3-turbo (fast + accurate). CPU: small is more practical.
STT_MODEL = "large-v3-turbo" if DEVICE == "cuda" else "small"
print(f"device={DEVICE} compute={COMPUTE} model={STT_MODEL}")

stt = WhisperModel(STT_MODEL, device=DEVICE, compute_type=COMPUTE)

def transcribe(path, language=None, task="transcribe", timestamps=False):
    """language=None -> auto-detect. task='translate' -> output English."""
    segments, info = stt.transcribe(
        path,
        language=language,
        task=task,
        vad_filter=True,          # skips silence, cuts hallucinations
        beam_size=5,
    )
    segs = list(segments)         # generator -> list (decoding happens here)
    text = " ".join(s.text.strip() for s in segs)
    if timestamps:
        text = "\n".join(f"[{s.start:6.1f}s → {s.end:6.1f}s] {s.text.strip()}" for s in segs)
    return text, info.language, info.language_probability

## 3. Text → Speech engine (Kokoro-82M)

In [ ]:
from kokoro import KPipeline

SR = 24000
VOICES = {  # lang_code -> example voice
    "a": "af_heart",   # American English
    "b": "bf_emma",    # British English
    "e": "ef_dora",    # Spanish
    "f": "ff_siwis",   # French
    "h": "hf_alpha",   # Hindi
    "i": "if_sara",    # Italian
    "j": "jf_alpha",   # Japanese
    "p": "pf_dora",    # Portuguese
    "z": "zf_xiaobei", # Mandarin
}
_pipes = {}

def synthesize(text, lang="a", voice=None, speed=1.0):
    if lang not in _pipes:
        _pipes[lang] = KPipeline(lang_code=lang)
    voice = voice or VOICES[lang]
    chunks = []
    for _, _, audio in _pipes[lang](text, voice=voice, speed=speed):
        a = audio.cpu().numpy() if hasattr(audio, "cpu") else np.asarray(audio)
        chunks.append(a)
    return np.concatenate(chunks).astype(np.float32)

from IPython.display import Audio, display
wav = synthesize("Hello! This is a speech system running inside Colab.")
display(Audio(wav, rate=SR))

## 4. Record from your microphone inside Colab

In [ ]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import subprocess

_JS = '''
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b2text = blob => new Promise(res => {
  const r = new FileReader(); r.onloadend = e => res(e.srcElement.result); r.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.onstop = async () => resolve(await b2text(new Blob(chunks)));
  rec.start();
  await sleep(time);
  rec.stop();
  stream.getTracks().forEach(t => t.stop());
});
'''

def record_audio(seconds=5, out="mic.wav"):
    display(Javascript(_JS))
    print(f"🔴 Recording {seconds}s ... speak now")
    data = output.eval_js(f"record({seconds * 1000})")
    open("mic.webm", "wb").write(b64decode(data.split(",")[1]))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "mic.webm",
                    "-ar", "16000", "-ac", "1", out], check=True)
    print("✅ done")
    return out

# Uncomment to try (browser will ask for mic permission):
# path = record_audio(6)
# print(transcribe(path))

## 5. Round-trip test: text → speech → text

In [ ]:
original = "The quick brown fox jumps over the lazy dog, then asks for a cup of coffee."
sf.write("rt.wav", synthesize(original), SR)

text, lang, prob = transcribe("rt.wav")
print("IN :", original)
print("OUT:", text)
print(f"lang={lang} ({prob:.0%})")

## 6. Gradio UI (mic / upload → text, and text → audio)

In [ ]:
import gradio as gr

def ui_stt(audio_path, language, task, timestamps):
    if audio_path is None:
        return "No audio provided.", ""
    lang = None if language == "auto" else language
    text, detected, p = transcribe(audio_path, lang, task, timestamps)
    return text, f"Detected: {detected} ({p:.0%})"

def ui_tts(text, lang, speed):
    if not text.strip():
        return None
    return (SR, synthesize(text, lang, speed=speed))

with gr.Blocks(title="Whisper Voice System") as demo:
    gr.Markdown("# 🎙️ Whisper Voice System")
    with gr.Tab("Speech → Text"):
        a = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Audio")
        with gr.Row():
            l = gr.Dropdown(["auto", "en", "sw", "fr", "es", "de", "ar", "hi", "zh", "ja"], value="auto", label="Language")
            t = gr.Radio(["transcribe", "translate"], value="transcribe", label="Task (translate → English)")
            ts = gr.Checkbox(label="Timestamps")
        btn = gr.Button("Transcribe", variant="primary")
        out, info = gr.Textbox(label="Transcript", lines=8), gr.Textbox(label="Info")
        btn.click(ui_stt, [a, l, t, ts], [out, info])
    with gr.Tab("Text → Speech"):
        txt = gr.Textbox(label="Text", lines=5, value="Hello from Colab!")
        with gr.Row():
            lg = gr.Dropdown(list(VOICES), value="a", label="Kokoro lang code")
            sp = gr.Slider(0.6, 1.4, value=1.0, label="Speed")
        b2 = gr.Button("Speak", variant="primary")
        o2 = gr.Audio(label="Output")
        b2.click(ui_tts, [txt, lg, sp], o2)

demo.launch(share=True, debug=False)

## 7. Optional extras

### 7a. Languages Kokoro doesn't cover (e.g. Swahili): Meta MMS-TTS
Whisper already **transcribes** Swahili. For **speech output**, use `facebook/mms-tts-swh`.

In [ ]:
from transformers import VitsModel, AutoTokenizer

mms_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-swh")
mms = VitsModel.from_pretrained("facebook/mms-tts-swh").to(DEVICE)

def synthesize_swahili(text):
    inputs = mms_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        wav = mms(**inputs).waveform[0].cpu().numpy()
    return wav, mms.config.sampling_rate

w, sr = synthesize_swahili("Habari yako, karibu kwenye mfumo wa sauti.")
display(Audio(w, rate=sr))

### 7b. Turn it into a FastAPI service
Wrap `transcribe()` / `synthesize()` in two endpoints (`POST /stt` takes multipart audio, `POST /tts` takes JSON and returns `audio/wav`).
Expose from Colab with `cloudflared` or `ngrok`, then containerize with Docker + CUDA base image for deployment.

### 7c. Next steps
- **Streaming/real-time**: WebSocket endpoint + chunked VAD (Silero) + faster-whisper per utterance
- **Voice agent**: STT → LangGraph agent → TTS loop
- **Fine-tune Whisper** on your own language/domain data (LoRA via `peft`)
- **Benchmark**: WER on your own recordings, `large-v3-turbo` vs `small` vs `distil-large-v3`